In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

df = pd.read_csv("../data/processed/secom_clean.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
sensor_cols = [c for c in df.columns if c.startswith("sensor_")]

X = df[sensor_cols]
y = (df["label"] == 1).astype(int)   # 1 = fail, 0 = pass

# split A: by time (first 70% of samples for training, last 30% for testing)
n_train = int(len(df) * 0.7)
X_train_t, X_test_t = X.iloc[:n_train], X.iloc[n_train:]
y_train_t, y_test_t = y.iloc[:n_train], y.iloc[n_train:]

# split B: random, keeping the same fail rate in both parts
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

print("Time split  - train:", len(y_train_t), "samples,", y_train_t.sum(), "fails | test:",
      len(y_test_t), "samples,", y_test_t.sum(), "fails")
print("Random split - train:", len(y_train_r), "samples,", y_train_r.sum(), "fails | test:",
      len(y_test_r), "samples,", y_test_r.sum(), "fails")
print("Test period for time split:", df["timestamp"].iloc[n_train], "to", df["timestamp"].iloc[-1])

Time split  - train: 1096 samples, 78 fails | test: 471 samples, 26 fails
Random split - train: 1096 samples, 73 fails | test: 471 samples, 31 fails
Test period for time split: 2008-09-26 03:12:00 to 2008-10-17 06:07:00


In [2]:
def evaluate(name, X_tr, y_tr, X_te, y_te):
    # the pipeline learns the median and scaling from the training data only,
    # then applies them to the test data (no leakage)
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=0.01, class_weight="balanced", max_iter=1000),
    )
    model.fit(X_tr, y_tr)
    scores = model.predict_proba(X_te)[:, 1]
    print(f"{name}: PR-AUC = {average_precision_score(y_te, scores):.3f} "
          f"(random guess = {y_te.mean():.3f}), ROC-AUC = {roc_auc_score(y_te, scores):.3f}")
    return model, scores

model_t, scores_t = evaluate("Time split  ", X_train_t, y_train_t, X_test_t, y_test_t)
model_r, scores_r = evaluate("Random split", X_train_r, y_train_r, X_test_r, y_test_r)

Time split  : PR-AUC = 0.070 (random guess = 0.055), ROC-AUC = 0.592
Random split: PR-AUC = 0.137 (random guess = 0.066), ROC-AUC = 0.684


In [3]:
def make_model():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=0.01, class_weight="balanced", max_iter=1000),
    )

# a time-only feature: days since the first sample
days = ((df["timestamp"] - df["timestamp"].min()).dt.total_seconds() / 86400).to_frame("days")

# repeat the random split 30 times: sensor model vs a model that only knows the date
rows = []
for seed in range(30):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)

    sensor_model = make_model().fit(Xtr, ytr)
    auc_sensors = roc_auc_score(yte, sensor_model.predict_proba(Xte)[:, 1])

    time_model = LogisticRegression(class_weight="balanced").fit(days.loc[Xtr.index], ytr)
    auc_time = roc_auc_score(yte, time_model.predict_proba(days.loc[Xte.index])[:, 1])

    rows.append((auc_sensors, auc_time))

res = pd.DataFrame(rows, columns=["sensors_auc", "time_only_auc"])
print(res.describe().round(3))

       sensors_auc  time_only_auc
count       30.000         30.000
mean         0.674          0.631
std          0.039          0.045
min          0.596          0.558
25%          0.657          0.597
50%          0.680          0.633
75%          0.701          0.664
max          0.741          0.714


## Random split vs time split
With a random split, the sensor model gets a ROC-AUC of 0.67 on average (30 repeats, std 0.04).
A model that only knows the date gets 0.63. So most of what the sensor model learns in a random split
could just be time. With a time split (train on early samples, test on the last 3 weeks) the ROC-AUC is 0.59
and the PR-AUC is 0.07, compared to 0.055 for random guessing. But that test set only has 26 failures,
so this number is very noisy. My takeaway: the sensors have only a weak link to failure
once time is taken out, and a random split makes the model look better than it really is.

In [4]:
res["diff"] = res["sensors_auc"] - res["time_only_auc"]
print("mean diff:", res["diff"].mean().round(3), "| std:", res["diff"].std().round(3))
print("sensor model better in", (res["diff"] > 0).sum(), "of 30 splits")

mean diff: 0.042 | std: 0.05
sensor model better in 26 of 30 splits


In [5]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier

def make_rf():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                               class_weight="balanced_subsample", n_jobs=-1, random_state=0),
    )

tscv = TimeSeriesSplit(n_splits=5)
rows = []
for k, (tr, te) in enumerate(tscv.split(X), start=1):
    ytr, yte = y.iloc[tr], y.iloc[te]
    if yte.sum() == 0:
        continue
    row = {"fold": k, "train_n": len(tr),
           "test_start": df["timestamp"].iloc[te].min().date(),
           "test_end": df["timestamp"].iloc[te].max().date(),
           "test_fails": int(yte.sum()), "base_pr": yte.mean()}
    for name, model in [("logreg", make_model()), ("forest", make_rf())]:
        model.fit(X.iloc[tr], ytr)
        p = model.predict_proba(X.iloc[te])[:, 1]
        row[f"{name}_roc"] = roc_auc_score(yte, p)
        row[f"{name}_pr"] = average_precision_score(yte, p)
    rows.append(row)

cv = pd.DataFrame(rows)
print(cv.round(3).to_string(index=False))

 fold  train_n test_start   test_end  test_fails  base_pr  logreg_roc  logreg_pr  forest_roc  forest_pr
    1      262 2008-08-18 2008-08-29          22    0.084       0.572      0.101       0.526      0.179
    2      523 2008-08-29 2008-09-11           6    0.023       0.554      0.134       0.505      0.027
    3      784 2008-09-11 2008-09-23          10    0.038       0.592      0.060       0.503      0.229
    4     1045 2008-09-23 2008-10-05          13    0.050       0.457      0.047       0.600      0.068
    5     1306 2008-10-05 2008-10-17          14    0.054       0.787      0.216       0.629      0.145


## Time-ordered cross-validation
I split the data into 5 time-ordered folds (always train on the past, test on the next period).
The ROC-AUC of the logistic regression ranges from 0.46 to 0.79 and averages about 0.59. The random forest
averages about 0.55, so it is not better. The test folds only have 6 to 22 failures each, so single numbers
jump around a lot and I don't trust the best or the worst fold. My conclusion: with these sensors alone,
early warning is weak, and I'll keep the simpler logistic regression instead of tuning more.

In [6]:
from scipy.stats import binomtest

def top_frac_caught(make_fn, frac=0.10):
    caught, total, flagged, n_test = 0, 0, 0, 0
    for tr, te in TimeSeriesSplit(n_splits=5).split(X):
        ytr, yte = y.iloc[tr], y.iloc[te]
        model = make_fn().fit(X.iloc[tr], ytr)
        p = model.predict_proba(X.iloc[te])[:, 1]
        k = int(np.ceil(frac * len(te)))
        top = np.argsort(-p)[:k]          # the k samples with the highest risk score
        caught += yte.iloc[top].sum()
        total += yte.sum()
        flagged += k
        n_test += len(te)
    return caught, total, flagged, n_test

for frac in [0.1, 0.2, 0.3]:
    c, t, f, n = top_frac_caught(make_model, frac)
    p = binomtest(int(c), int(t), f / n, alternative="greater").pvalue
    print(f"flag top {frac:.0%}: caught {c} of {t} failures ({c/t:.0%}); "
          f"random flagging would catch about {f/n:.0%}; p = {p:.3f}")

flag top 10%: caught 12 of 65 failures (18%); random flagging would catch about 10%; p = 0.033
flag top 20%: caught 18 of 65 failures (28%); random flagging would catch about 20%; p = 0.095
flag top 30%: caught 27 of 65 failures (42%); random flagging would catch about 30%; p = 0.035
